In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error

df = pd.read_csv('../../data/final/df_final_model.csv')

features = [
    'tree_density_per_km2',
    'light_density_per_km2',
    'mean_dist_nightlife',
    'mean_dist_mobility',
    'mean_dist_retail',
    'mean_dist_education',
    'mean_dist_culture_sport',
    'mean_dist_emergency_health',
    'population'
]

target = 'log_crime_rate'

#Preprocessing
df_model = df.dropna(subset=features + [target])

X = df_model[features]
y = df_model[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

#Ridge Regression with Cross-Validation
ridge = RidgeCV(alphas=np.logspace(-6, 6, 13))
ridge.fit(X_train_scaled, y_train)

# Evaluation
y_pred = ridge.predict(X_test_scaled)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"--- Model Performance ---")
print(f"R² Score: {r2:.3f} (Explains {r2*100:.1f}% of the variance)")
print(f"RMSE: {rmse:.3f}")
print(f"Optimal Regularization (Alpha): {ridge.alpha_}")

# Interpret Coefficients
coef_df = pd.DataFrame({
    'Feature': features,
    'Coefficient': ridge.coef_
}).sort_values(by='Coefficient', ascending=False)

print("\n--- Feature Importance (Impact on Crime Rate) ---")
print(coef_df)

# Plot
colors = ['#ef5350' if x > 0 else '#42a5f5' for x in coef_df['Coefficient']]

plt.figure(figsize=(10, 6))

# 2. Gebruik 'palette=colors' om de kleuren direct toe te wijzen
sns.barplot(x='Coefficient', y='Feature', data=coef_df, palette=colors)

plt.axvline(0, color='black', linestyle='--')
plt.title('How spatial Features Predict Crime (Ridge Coefficients)')
plt.show()

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from esda.moran import Moran
from libpysal.weights import Queen


df = pd.read_csv('../../data/final/df_final_model.csv')

gdf_all = gpd.read_file("../../data/raw/boundaries/wijken_en_gemeenten.gpkg")
gdf_ams = gdf_all[gdf_all['gemeentenaam'] == 'Amsterdam'].copy()

gdf_merged = gdf_ams.merge(df, left_on='wijkcode', right_on='neighbourhood_code')


features = ['tree_density_per_km2', 'light_density_per_km2', 'mean_dist_nightlife',
            'mean_dist_mobility', 'mean_dist_retail', 'mean_dist_education',
            'mean_dist_culture_sport', 'mean_dist_emergency_health', 'population']
target = 'log_crime_rate'

gdf_model = gdf_merged.dropna(subset=features + [target])


scaler = StandardScaler()
X = gdf_model[features]
y = gdf_model[target]
X_scaled = scaler.fit_transform(X)

ridge = RidgeCV(alphas=np.logspace(-6, 6, 13))
ridge.fit(X_scaled, y)

r2_score = ridge.score(X_scaled, y)
print(f"--- Model Fit ---")
print(f"R² Score: {r2_score:.3f}")

# Calculate Residuals (Errors)
gdf_model['predicted'] = ridge.predict(X_scaled)
gdf_model['residual'] = gdf_model[target] - gdf_model['predicted']


# Create Spatial Weights
w = Queen.from_dataframe(gdf_model)
w.transform = 'r'

# Calculate Moran's I on the Residuals
mi = Moran(gdf_model['residual'], w)

print(f"Moran's I: {mi.I:.3f}")
print(f"P-value: {mi.p_sim:.3f}")

if mi.p_sim < 0.05:
    print("Result: Significant Clustering (The model missed a spatial pattern)")
else:
    print("Result: Random Errors (The model works well spatially!)")

# Plot the Map of Residuals
fig, ax = plt.subplots(1, 1, figsize=(12, 10))
gdf_model.plot(column='residual', cmap='RdBu', legend=True, ax=ax,
               legend_kwds={'label': "Model Error (Red=Underpredicted, Blue=Overpredicted)"})
ax.set_title('Spatial Clustering of Prediction Errors')
ax.axis('off')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd

# Assuming you still have 'gdf_model' from the previous step
# If not, re-run the previous code block to create it

fig, axes = plt.subplots(1, 3, figsize=(20, 8))

# --- Map 1: Actual Crime (Log Scale) ---
gdf_model.plot(column='log_crime_rate',
               cmap='OrRd',
               scheme='quantiles',
               k=5,
               legend=True,
               ax=axes[0])
axes[0].set_title('A. Actual Crime (Log)', fontsize=14)
axes[0].axis('off')

# --- Map 2: Predicted Crime ---
gdf_model.plot(column='predicted',
               cmap='OrRd',
               scheme='quantiles', # Using same scheme for fair comparison
               k=5,
               legend=True,
               ax=axes[1])
axes[1].set_title('B. Predicted by Environment', fontsize=14)
axes[1].axis('off')

# --- Map 3: Residuals (The Gap) ---
gdf_model.plot(column='residual',
               cmap='RdBu_r', # Red = Underpredicted (Danger), Blue = Overpredicted (Safe)
               legend=True,
               ax=axes[2])
axes[2].set_title('C. Residuals (Unexplained)', fontsize=14)
axes[2].axis('off')

plt.suptitle('Comparison: Reality vs. Model vs. Errors', fontsize=20)
plt.tight_layout()
plt.show()